In [14]:
from modelscope import AutoModelForCausalLM, AutoTokenizer

from utils import SYSTEM_PROMPT

In [16]:
model_id = "qwen/Qwen2.5-0.5B-Instruct"  

In [25]:

output="outputs/Qwen-0.5B-SFT-FirstHalf/checkpoint-234"
model = AutoModelForCausalLM.from_pretrained(
        output,
        torch_dtype="auto",
        device_map="auto"
    )
tokenizer = AutoTokenizer.from_pretrained(output)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [26]:
prompt="Xiao dudu bought 4 apples, ate 1, and gave 1 to his sister. How many apples were left?"

In [27]:
messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]

In [28]:
text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )
generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [29]:
response

'<think>\nInitially, Xiao dudu had 4 apples.\nHe ate 1, so he now has \\(4 - 1 = <<4-1=3>>3\\) apples.\nThen, he gave 1 to his sister, leaving him with \\(3 - 1 = \\boxed{2}\\) apples.\n</think>\n<answer>\n2\n</answer>\n'